# MemeGPT — Remote Ollama Server on Colab T4
**Runtime → Change runtime type → T4 GPU** before running.

Run all cells in order. Copy the `OLLAMA_HOST` URL printed at the end
and paste it into your local `.env` or `docker-compose.yml`.

Uses ngrok **HTTP** tunnel (free, no credit card required).

In [ ]:
# Cell 1 — Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Cell 2 — Start Ollama server with open CORS (needed for ngrok HTTP tunnel)
import subprocess, time, os

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
env['OLLAMA_ORIGINS'] = '*'   # allow requests from any ngrok subdomain

proc = subprocess.Popen(
    ['ollama', 'serve'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)
print('Ollama server started (pid', proc.pid, ')')

In [ ]:
# Cell 3 — Pull the model
!ollama pull llama3.1:8b

In [ ]:
# Cell 4 — Expose Ollama via ngrok HTTP tunnel (free tier, no card required)
# Get a free token at https://dashboard.ngrok.com/signup
NGROK_TOKEN = ''  # <-- paste your free ngrok authtoken here

!pip install -q pyngrok
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# HTTP tunnel — free tier, no credit card needed (TCP requires card, HTTP doesn't)
tunnel = ngrok.connect(11434)   # defaults to http
host = tunnel.public_url        # e.g. https://xxxx.ngrok-free.app

print('=' * 60)
print(f'OLLAMA_HOST={host}')
print('=' * 60)
print()
print('On your local machine:')
print('  ./scripts/use_remote_ollama.sh', host)
print()
print('Or for native dev (no Docker):')
print(f'  OLLAMA_HOST={host} uvicorn main:app --reload')

In [ ]:
# Cell 5 — Keep alive (prevents Colab from disconnecting)
import time
print('Server running. Keep this cell active (do not close the tab).')
while True:
    time.sleep(30)